####Crear la tabla results_movie en la capa "gold"

In [0]:
# %sql
# USE movie_silver;

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
# %sql
# CREATE TABLE movie_gold.results_movie
# USING DELTA
# AS
# SELECT M.year_Release_Date, C.country_Name, PCO.company_Name, M.budget, M.revenue
# FROM movies M
# INNER JOIN production_country PC ON M.movie_Id = PC.movie_Id
# INNER JOIN countries C ON PC.country_Id = C.country_Id
# INNER JOIN movies_companies MC ON M.movie_Id = MC.movie_Id
# INNER JOIN productions_companies PCO ON MC.company_Id = PCO.company_Id;

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS movie_gold.results_movie
          (
              year_Release_Date INT,
              country_Name STRING,
              company_Name STRING,
              budget FLOAT,
              revenue FLOAT,
              movie_id INT,
              country_id INT,
              company_id INT,
              created_date date,
              updated_date date              
          )
          USING DELTA
          """
         )

DataFrame[]

In [0]:
spark.sql(f"""
            CREATE OR REPLACE TEMP VIEW V_results_movie
            AS
            SELECT M.year_Release_Date, C.country_Name, PCO.company_Name, M.budget, M.revenue, M.movie_Id, C.country_Id, PCO.company_Id
            FROM movie_silver.movies M
            INNER JOIN movie_silver.production_country PC ON M.movie_Id = PC.movie_Id
            INNER JOIN movie_silver.countries C ON PC.country_Id = C.country_Id
            INNER JOIN movie_silver.movies_companies MC ON M.movie_Id = MC.movie_Id
            INNER JOIN movie_silver.productions_companies PCO ON MC.company_Id = PCO.company_Id
            WHERE M.file_date = '{v_file_date}'
        """)

DataFrame[]

In [0]:
%sql
SELECT * FROM V_results_movie;

year_Release_Date,country_Name,company_Name,budget,revenue,movie_Id,country_Id,company_Id
2012,United States of America,Spy Global Media,500000.0,625000.0,117942,214,31296
2014,United States of America,Moving Picture Company (MPC),1.7E8,7.73328629E8,118340,214,20478
2014,United Kingdom,Moving Picture Company (MPC),1.7E8,7.73328629E8,118340,162,20478
2014,United States of America,Marvel Studios,1.7E8,7.73328629E8,118340,214,420
2014,United Kingdom,Marvel Studios,1.7E8,7.73328629E8,118340,162,420
2014,United States of America,Bulletproof Cupid,1.7E8,7.73328629E8,118340,214,54850
2014,United Kingdom,Bulletproof Cupid,1.7E8,7.73328629E8,118340,162,54850
2014,United States of America,Revolution Sun Studios,1.7E8,7.73328629E8,118340,214,76043
2014,United Kingdom,Revolution Sun Studios,1.7E8,7.73328629E8,118340,162,76043
1998,United States of America,Forensic Films,300000.0,40542.0,118452,214,2813


In [0]:
spark.sql("""
            MERGE INTO movie_gold.results_movie tgt
            USING V_results_movie src
            ON (tgt.movie_id = src.movie_Id AND tgt.country_id = src.country_Id AND tgt.company_id = src.company_Id) 
            WHEN MATCHED THEN
                UPDATE SET
                    tgt.year_Release_Date = src.year_Release_Date,
                    tgt.country_Name = src.country_Name,
                    tgt.company_Name = src.company_Name,
                    tgt.budget = src.budget,
                    tgt.revenue = src.revenue,
                    tgt.updated_date = current_timestamp
            WHEN NOT MATCHED THEN
                INSERT (year_Release_Date, country_Name, company_Name, budget, revenue, movie_id, country_id, company_id, created_date) 
                VALUES (year_Release_Date, country_Name, company_Name, budget, revenue, movie_Id, country_Id, company_Id, current_timestamp)
         """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
SELECT * FROM movie_gold.results_movie;

year_Release_Date,country_Name,company_Name,budget,revenue,movie_id,country_id,company_id,created_date,updated_date
2008,United States of America,DreamWorks Animation,1.5E8,6.0390035E8,10527,214,521,2026-09-15,2026-09-15
2009,Germany,Village Roadshow Pictures,9.0E7,5.24028672E8,10528,151,79,2026-09-15,2026-09-15
2009,Germany,Warner Bros.,9.0E7,5.24028672E8,10528,151,6194,2026-09-15,2026-09-15
2009,Germany,Wigram Productions,9.0E7,5.24028672E8,10528,151,23202,2026-09-15,2026-09-15
2009,Germany,Internationale Filmproduktion Blackbird Dritte,9.0E7,5.24028672E8,10528,151,19855,2026-09-15,2026-09-15
2009,Germany,Silver Pictures,9.0E7,5.24028672E8,10528,151,1885,2026-09-15,2026-09-15
2009,United Kingdom,Village Roadshow Pictures,9.0E7,5.24028672E8,10528,162,79,2026-09-15,2026-09-15
2009,United Kingdom,Warner Bros.,9.0E7,5.24028672E8,10528,162,6194,2026-09-15,2026-09-15
2009,United Kingdom,Wigram Productions,9.0E7,5.24028672E8,10528,162,23202,2026-09-15,2026-09-15
2009,United Kingdom,Internationale Filmproduktion Blackbird Dritte,9.0E7,5.24028672E8,10528,162,19855,2026-09-15,2026-09-15


In [0]:
%sql
SELECT COUNT(1) FROM V_results_movie;

count(1)
3247


In [0]:
%sql
SELECT COUNT(1) FROM movie_gold.results_movie;

count(1)
21965
